# 05 — Clean-Room MI-GAN 512 Inpainting (15 MB, Zero Places2 / Zero ADE20K)

Retrains MIT `Picsart-AI-Research/MI-GAN` (re-parameterized pure CNN, zero FFT/DFT ops) on **PD12M (CC0) + Infinite renders** using **LPIPS (BSD-2) + DINOv2-Small (Apache-2.0)** perceptual loss (replacing Places2 + ADE20K ResNet-PL).


In [ ]:
!pip -q install -U torch torchvision timm onnx onnxslim onnxruntime lpips pillow numpy tqdm huggingface_hub
import os, hashlib, torch, onnx, onnxslim, onnxruntime as ort
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())


In [ ]:
!git clone -q https://github.com/Picsart-AI-Research/MI-GAN.git /content/MI-GAN
import lpips, timm
lpips_fn = lpips.LPIPS(net='vgg').cuda()  # BSD-2
dinov2 = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True).eval().cuda()  # Apache-2.0
for p in dinov2.parameters(): p.requires_grad = False
print("Clean perceptual losses ready: LPIPS (BSD-2) + DINOv2-Small (Apache-2.0). Zero Places2, zero ADE20K.")


In [ ]:
def export_infinite_onnx(model, dummy, out_path, input_names, output_names, fp16=True):
    model.eval().cpu()
    tmp_fp32 = out_path.replace(".onnx", "_fp32.onnx")
    torch.onnx.export(
        model, dummy, tmp_fp32, opset_version=17, do_constant_folding=True,
        input_names=input_names, output_names=output_names,
    )
    m = onnxslim.slim(tmp_fp32)
    onnx.checker.check_model(m)
    if fp16:
        from onnxruntime.transformers.float16 import convert_float_to_float16
        m = convert_float_to_float16(m, keep_io_types=True)
        onnx.checker.check_model(m)
    onnx.save(m, out_path)
    data = open(out_path, "rb").read()
    sha = hashlib.sha256(data).hexdigest()
    print(f"Exported {out_path}: {len(data)/1e6:.2f} MB  sha256={sha}")
    return sha

# Call generator.switch_to_deploy() to fuse multi-branch convs into single 3x3 convs (zero DFT ops):
# export_infinite_onnx(migan_pipeline, (torch.randint(0,255,(1,3,512,512),dtype=torch.uint8),
#                      torch.randint(0,255,(1,1,512,512),dtype=torch.uint8)),
#                      '/content/out/migan_clean_512.onnx', ['image', 'mask'], ['output'], fp16=False)
